In [ ]:
# =============================================================
# CAViaR — ASYMMETRIC SLOPE
# Engle & Manganelli (2004)
# VaR_t = β₀ + β₁·VaR_{t-1} + β₂·max(r_{t-1},0) + β₃·min(r_{t-1},0)
#
# Protocolo idéntico al resto de modelos del TFG:
#   - Ventana rolling 900 obs, reestimación diaria
#   - Optimización: Nelder-Mead (scipy)
#   - Inicialización: β=[0.01, 0.9, 0.1, 0.1]
#   - VaR inicial: cuantil p99 empírico de las primeras 100 obs
# =============================================================

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import chi2
import time
import warnings
warnings.filterwarnings("ignore")

# ── Datos ──────────────────────────────────────────────────────
dataset = pd.read_pickle("../data/dataset_tfg.pkl").copy().sort_index()
alpha   = 0.99

print(f"Dataset: {dataset.shape[0]} obs  ({dataset.index[0].date()} → {dataset.index[-1].date()})")
print(f"Columna objetivo: target  |  μ={dataset['target'].mean():.5f}  σ={dataset['target'].std():.5f}")

In [ ]:
# =============================================================
# IMPLEMENTACIÓN CAViaR-AS
# =============================================================

def caviar_seq(params, r):
    """Genera la secuencia VaR_{1..T} con la recursión CAViaR-AS."""
    b0, b1, b2, b3 = params
    T   = len(r)
    VaR = np.empty(T)
    VaR[0] = np.quantile(r[:100], alpha)   # arranque: p99 empírico de las primeras 100 obs
    for t in range(1, T):
        rp     = r[t - 1]
        VaR[t] = b0 + b1 * VaR[t-1] + b2 * max(rp, 0.0) + b3 * min(rp, 0.0)
    return VaR


def tick_loss_obj(params, r):
    """Tick Loss (Quantile Loss al 99%) sobre la secuencia de entrenamiento."""
    VaR = caviar_seq(params, r)
    e   = r - VaR
    return float(np.mean(np.where(e >= 0.0, alpha * e, (alpha - 1.0) * e)))


def fit_caviar(r_train):
    """Estima β minimizando la Tick Loss con Nelder-Mead."""
    res = minimize(
        tick_loss_obj,
        x0=[0.01, 0.9, 0.1, 0.1],
        args=(r_train,),
        method="Nelder-Mead",
        options={"maxiter": 3000, "xatol": 1e-6, "fatol": 1e-7, "adaptive": True},
    )
    return res.x


def predict_next(params, r_train):
    """VaR para el día siguiente al último de la ventana de entrenamiento."""
    b0, b1, b2, b3 = params
    VaR_T = caviar_seq(params, r_train)[-1]   # último VaR de la secuencia ajustada
    rp    = r_train[-1]                        # última pérdida observada
    return b0 + b1 * VaR_T + b2 * max(rp, 0.0) + b3 * min(rp, 0.0)


print("Funciones CAViaR-AS definidas.")

In [ ]:
# =============================================================
# EVALUACIÓN ROLLING — 2015-2024
# =============================================================

eval_start = pd.Timestamp("2015-01-01")
eval_end   = pd.Timestamp("2024-12-31")

eval_dates = dataset.loc[eval_start:eval_end].index
N = len(eval_dates)
print(f"Período de evaluación : {eval_start.date()} → {eval_end.date()}")
print(f"Días de evaluación    : {N}")
print()

var_preds    = []
real_targets = []
dates        = []
t_per_day    = []

t_total_start = time.perf_counter()

for i, current_date in enumerate(eval_dates):
    t0 = time.perf_counter()

    train_end = current_date - pd.Timedelta(days=1)
    train_df  = dataset.loc[:train_end].tail(900)
    r_train   = train_df["target"].dropna().values

    if len(r_train) < 250:
        continue

    params    = fit_caviar(r_train)
    var_pred  = predict_next(params, r_train)
    test_loss = dataset.loc[current_date, "target"]

    var_preds.append(var_pred)
    real_targets.append(test_loss)
    dates.append(current_date)

    dt = time.perf_counter() - t0
    t_per_day.append(dt)

    if (i + 1) % 100 == 0 or i == 0:
        elapsed = time.perf_counter() - t_total_start
        eta     = (elapsed / (i + 1)) * (N - i - 1)
        print(f"  [{i+1:4d}/{N}] {current_date.date()} | "
              f"VaR={var_pred:.5f} | loss={test_loss:.5f} | "
              f"{dt:.3f}s/día | ETA {eta/60:.1f}min")

t_total = time.perf_counter() - t_total_start
t_mean  = float(np.mean(t_per_day))

caviar_df = pd.DataFrame(
    {"VaR_CAViaR": var_preds, "loss_real": real_targets},
    index=pd.to_datetime(dates),
)
print(f"\nResultados: {len(caviar_df)} días | total {t_total/60:.1f} min | {t_mean:.3f} s/día")

In [ ]:
# =============================================================
# TIMING
# =============================================================

n_full   = len(dataset.loc["2015":"2024"])
t_est_s  = t_mean * n_full

SEP = "=" * 56
print(SEP)
print("TIMING")
print(SEP)
print(f"  Días procesados       : {len(caviar_df)}")
print(f"  Tiempo total (2015)   : {t_total:.1f} s  ({t_total/60:.1f} min)")
print(f"  Tiempo medio / día    : {t_mean:.3f} s")
print(f"  Días en 2015-2024     : {n_full}")
print(f"  Estimación 2015-2024  : {t_est_s:.0f} s  "
      f"({t_est_s/60:.1f} min  /  {t_est_s/3600:.1f} h)")

In [ ]:
# =============================================================
# FUNCIONES DE BACKTESTING (idénticas a comp_final)
# =============================================================

def hit_series(rl, vp):
    rl = np.asarray(rl, dtype=float).ravel()
    vp = np.asarray(vp, dtype=float).ravel()
    m  = np.isfinite(rl) & np.isfinite(vp)
    return rl[m], vp[m], (rl[m] > vp[m]).astype(int)

def kupiec_test(rl, vp):
    _, _, I = hit_series(rl, vp)
    T, x   = len(I), int(I.sum())
    p = 1 - alpha
    if T == 0 or x == 0 or x == T:
        return np.nan, np.nan
    ph  = x / T
    LR  = -2 * np.log(((1-p)**(T-x) * p**x) / ((1-ph)**(T-x) * ph**x))
    return LR, 1 - chi2.cdf(LR, 1)

def cc_test(rl, vp):
    _, _, I = hit_series(rl, vp)
    n00 = n01 = n10 = n11 = 0
    for t in range(1, len(I)):
        pair = (I[t-1], I[t])
        if   pair == (0,0): n00 += 1
        elif pair == (0,1): n01 += 1
        elif pair == (1,0): n10 += 1
        else:               n11 += 1
    if (n00+n01) == 0 or (n10+n11) == 0:
        return np.nan, np.nan
    pi01 = n01/(n00+n01);  pi11 = n11/(n10+n11)
    pi   = (n01+n11)/(n00+n01+n10+n11)
    L0   = (1-pi)**(n00+n10) * pi**(n01+n11)
    L1   = (1-pi01)**n00 * pi01**n01 * (1-pi11)**n10 * pi11**n11
    if L0 <= 0 or L1 <= 0:
        return np.nan, np.nan
    LRind    = -2*np.log(L0/L1)
    LRuc, _  = kupiec_test(rl, vp)
    if np.isnan(LRuc):
        return np.nan, np.nan
    return LRuc + LRind, 1 - chi2.cdf(LRuc + LRind, 2)

def dq_test(rl, vp, lags=4):
    _, _, I = hit_series(rl, vp)
    T = len(I);  p = 1 - alpha
    if T <= lags+5:
        return np.nan, np.nan
    hit = I - p
    y   = hit[lags:]
    X   = np.column_stack([np.ones(len(y))] + [hit[lags-j:T-j] for j in range(1, lags+1)])
    XtX = X.T @ X
    if np.linalg.matrix_rank(XtX) < XtX.shape[0]:
        return np.nan, np.nan
    beta   = np.linalg.inv(XtX) @ X.T @ y
    sigma2 = ((y - X @ beta)**2).mean()
    if sigma2 <= 0:
        return np.nan, np.nan
    DQ = float((beta @ XtX @ beta) / sigma2)
    return DQ, 1 - chi2.cdf(DQ, X.shape[1])

def tick_loss_metric(rl, vp):
    rl, vp, _ = hit_series(rl, vp)
    e = rl - vp
    return float(np.mean(np.where(e >= 0, alpha*e, (alpha-1)*e)))

def winkler_score(rl, vp):
    rl, vp, _ = hit_series(rl, vp)
    return float(np.mean(vp + (2/alpha) * np.maximum(rl - vp, 0)))

def eval_model(name, rl, vp):
    _, _, I  = hit_series(rl, vp)
    T, x     = len(I), int(I.sum())
    vr       = x / T
    _, p_uc  = kupiec_test(rl, vp)
    _, p_cc  = cc_test(rl, vp)
    _, p_dq  = dq_test(rl, vp)
    vals     = [v for v in [p_uc, p_cc, p_dq] if not np.isnan(v)]
    p_mean   = float(np.mean(vals)) if vals else np.nan
    return {
        "Model": name,  "T": T,
        "Violations": x,  "Exp. Viol.": round((1-alpha)*T, 2),
        "Violation Rate": vr,
        "p_UC": p_uc,  "UC": "PASS" if (not np.isnan(p_uc) and p_uc > 0.05) else "FAIL",
        "p_CC": p_cc,  "CC": "PASS" if (not np.isnan(p_cc) and p_cc > 0.05) else "FAIL",
        "p_DQ": p_dq,  "DQ": "PASS" if (not np.isnan(p_dq) and p_dq > 0.05) else "FAIL",
        "p_Mean": p_mean,
        "Tick Loss": tick_loss_metric(rl, vp),
        "Winkler Score": winkler_score(rl, vp),
        "Mean VaR Level": float(np.nanmean(vp)),
    }

print("Funciones de backtesting definidas.")

In [ ]:
# =============================================================
# BACKTESTING CAViaR-AS
# =============================================================

res_cav = eval_model("CAViaR-AS", caviar_df["loss_real"].values, caviar_df["VaR_CAViaR"].values)

fmt = {
    "T": lambda v: str(int(v)),
    "Violations": lambda v: str(int(v)),
    "Exp. Viol.": lambda v: f"{v:.2f}",
    "Violation Rate": lambda v: f"{v:.4f}",
    "p_UC": lambda v: f"{v:.4f}",
    "p_CC": lambda v: f"{v:.4f}",
    "p_DQ": lambda v: f"{v:.4f}",
    "p_Mean": lambda v: f"{v:.4f}",
    "Tick Loss": lambda v: f"{v:.6f}",
    "Winkler Score": lambda v: f"{v:.6f}",
    "Mean VaR Level": lambda v: f"{v:.5f}",
}

SEP = "=" * 56
print(SEP)
print("BACKTESTING CAViaR-AS — período de evaluación")
print(SEP)
for k, v in res_cav.items():
    if k == "Model":
        continue
    fv = "nan" if (isinstance(v, float) and np.isnan(v)) else (fmt[k](v) if k in fmt else str(v))
    print(f"  {k:<22}: {fv}")

In [ ]:
# =============================================================
# COMPARATIVA: CAViaR-AS  vs  MLP-VaR (w=20)  — mismo período
# =============================================================

mlp_df  = pd.read_csv("../data/nn_var_predictions_20.csv", index_col=0, parse_dates=True)

# Filtrar al mismo período y fechas comunes
period_start = caviar_df.index[0]
period_end   = caviar_df.index[-1]
mlp_period   = mlp_df.loc[period_start:period_end].dropna()

idx      = caviar_df.index.intersection(mlp_period.index)
cav_sub  = caviar_df.loc[idx]
mlp_sub  = mlp_period.loc[idx]

res_mlp = eval_model(
    "MLP-VaR (w=20)",
    mlp_sub["loss_real"].values,
    mlp_sub["VaR_pred"].values,
)

# Tabla lado a lado
metrics = [
    "T", "Violations", "Exp. Viol.", "Violation Rate",
    "p_UC", "UC", "p_CC", "CC", "p_DQ", "DQ", "p_Mean",
    "Tick Loss", "Winkler Score", "Mean VaR Level",
]

def fv(v, m):
    if isinstance(v, float) and np.isnan(v):
        return "nan"
    return fmt[m](v) if m in fmt else str(v)

SEP2 = "=" * 64
print(SEP2)
print(f"COMPARATIVA {period_start.year}–{period_end.year}: CAViaR-AS  vs  MLP-VaR (w=20)")
print(SEP2)
print(f"  {'Métrica':<22} {'CAViaR-AS':>18} {'MLP-VaR (w=20)':>18}")
print(f"  {'-'*22} {'-'*18} {'-'*18}")
for m in metrics:
    sc = fv(res_cav.get(m, np.nan), m)
    sm = fv(res_mlp.get(m, np.nan), m)
    print(f"  {m:<22} {sc:>18} {sm:>18}")

In [ ]:
# =============================================================
# GUARDAR PREDICCIONES
# =============================================================

caviar_df.to_csv("../data/caviar_var_predictions.csv")
print(f"Guardado → ../data/caviar_var_predictions.csv")
print(f"  {len(caviar_df)} filas  |  {caviar_df.index[0].date()} → {caviar_df.index[-1].date()}")
print(f"  VaR_CAViaR: min={caviar_df['VaR_CAViaR'].min():.5f}  "
      f"mean={caviar_df['VaR_CAViaR'].mean():.5f}  max={caviar_df['VaR_CAViaR'].max():.5f}")

In [ ]:
# =============================================================
# DESGLOSE ANUAL — CAViaR-AS
# =============================================================

def annual_metrics(df, var_col, loss_col):
    rows = []
    for year, g in df.groupby(df.index.year):
        rl = g[loss_col].values
        vp = g[var_col].values
        _, _, I = hit_series(rl, vp)
        T, x    = len(I), int(I.sum())
        vr      = x / T
        _, p_uc = kupiec_test(rl, vp)
        _, p_cc = cc_test(rl, vp)
        _, p_dq = dq_test(rl, vp)
        vals    = [v for v in [p_uc, p_cc, p_dq] if not np.isnan(v)]
        p_mean  = float(np.mean(vals)) if vals else np.nan
        rows.append({
            "Year":           year,
            "T":              T,
            "Viol":           x,
            "Exp":            round((1 - alpha) * T, 2),
            "VR":             vr,
            "p_UC":           p_uc,
            "UC":             "PASS" if (not np.isnan(p_uc) and p_uc > 0.05) else "FAIL",
            "p_CC":           p_cc,
            "CC":             "PASS" if (not np.isnan(p_cc) and p_cc > 0.05) else "FAIL",
            "p_DQ":           p_dq,
            "DQ":             "PASS" if (not np.isnan(p_dq) and p_dq > 0.05) else "FAIL",
            "p_Mean":         p_mean,
            "Tick Loss":      tick_loss_metric(rl, vp),
            "Winkler":        winkler_score(rl, vp),
            "Mean VaR":       float(np.nanmean(vp)),
        })
    return pd.DataFrame(rows).set_index("Year")

ann = annual_metrics(caviar_df, "VaR_CAViaR", "loss_real")

SEP = "=" * 110
print(SEP)
print("DESGLOSE ANUAL — CAViaR-AS (2015-2024)")
print(SEP)
print(f"{'Year':>4}  {'T':>4}  {'Viol':>4}  {'Exp':>4}  {'VR':>6}  "
      f"{'p_UC':>6}  {'UC':>4}  {'p_CC':>6}  {'CC':>4}  {'p_DQ':>6}  {'DQ':>4}  "
      f"{'p_Mean':>6}  {'TickLoss':>10}  {'Winkler':>10}  {'MeanVaR':>9}")
print("-" * 110)
for yr, r in ann.iterrows():
    puc  = f"{r['p_UC']:.3f}"  if not np.isnan(r['p_UC'])  else " nan"
    pcc  = f"{r['p_CC']:.3f}"  if not np.isnan(r['p_CC'])  else " nan"
    pdq  = f"{r['p_DQ']:.3f}"  if not np.isnan(r['p_DQ'])  else " nan"
    pm   = f"{r['p_Mean']:.3f}" if not np.isnan(r['p_Mean']) else " nan"
    print(f"{yr:>4}  {int(r['T']):>4}  {int(r['Viol']):>4}  {r['Exp']:>4.2f}  {r['VR']:>6.4f}  "
          f"{puc:>6}  {r['UC']:>4}  {pcc:>6}  {r['CC']:>4}  {pdq:>6}  {r['DQ']:>4}  "
          f"{pm:>6}  {r['Tick Loss']:>10.6f}  {r['Winkler']:>10.6f}  {r['Mean VaR']:>9.5f}")
print("-" * 110)
# Totales
rl_all = caviar_df["loss_real"].values
vp_all = caviar_df["VaR_CAViaR"].values
_, _, I_all = hit_series(rl_all, vp_all)
print(f"{'TOTAL':>4}  {len(I_all):>4}  {int(I_all.sum()):>4}  {(1-alpha)*len(I_all):>4.2f}  "
      f"{I_all.mean():>6.4f}  {'':>6}  {'':>4}  {'':>6}  {'':>4}  {'':>6}  {'':>4}  "
      f"{'':>6}  {tick_loss_metric(rl_all,vp_all):>10.6f}  "
      f"{winkler_score(rl_all,vp_all):>10.6f}  {np.nanmean(vp_all):>9.5f}")

In [ ]:
# =============================================================
# COMPARATIVA COMPLETA — 5 MODELOS (período completo 2015-2024)
# CAViaR-AS | MLP-VaR w=20 | GARCH-t | HS | GARCH-N
# =============================================================

mlp  = pd.read_csv("../data/nn_var_predictions_20.csv",   index_col=0, parse_dates=True)
hs   = pd.read_csv("../data/hs_var_predictions.csv",       index_col=0, parse_dates=True)
ga_t = pd.read_csv("../data/garch_t_var_predictions.csv",  index_col=0, parse_dates=True)
ga_n = pd.read_csv("../data/garch_n_var_predictions.csv",  index_col=0, parse_dates=True)
cav  = caviar_df.rename(columns={"VaR_CAViaR": "VaR_cav"})

# Fechas comunes a todos los modelos
idx = (cav.index
       .intersection(mlp.index)
       .intersection(hs.index)
       .intersection(ga_t.index)
       .intersection(ga_n.index))

cav  = cav.loc[idx]
mlp  = mlp.loc[idx]
hs   = hs.loc[idx]
ga_t = ga_t.loc[idx]
ga_n = ga_n.loc[idx]

print(f"Fechas comunes: {len(idx)}  ({idx[0].date()} → {idx[-1].date()})")

models_data = {
    "CAViaR-AS":      (cav["loss_real"].values,   cav["VaR_cav"].values),
    "MLP-VaR (w=20)": (mlp["loss_real"].values,   mlp["VaR_pred"].values),
    "GARCH-t":        (ga_t["loss_real"].values,   ga_t["VaR_GARCH"].values),
    "HS":             (hs["loss_real"].values,      hs["VaR_HS"].values),
    "GARCH-N":        (ga_n["loss_real"].values,    ga_n["VaR_GARCH_n"].values),
}

results = {name: eval_model(name, rl, vp) for name, (rl, vp) in models_data.items()}

metrics_cmp = [
    ("T",              lambda v: str(int(v))),
    ("Violations",     lambda v: str(int(v))),
    ("Exp. Viol.",     lambda v: f"{v:.2f}"),
    ("Violation Rate", lambda v: f"{v:.4f}"),
    ("p_UC",           lambda v: f"{v:.3f}" if not np.isnan(v) else "nan"),
    ("UC",             str),
    ("p_CC",           lambda v: f"{v:.3f}" if not np.isnan(v) else "nan"),
    ("CC",             str),
    ("p_DQ",           lambda v: f"{v:.3f}" if not np.isnan(v) else "nan"),
    ("DQ",             str),
    ("p_Mean",         lambda v: f"{v:.3f}" if not np.isnan(v) else "nan"),
    ("Tick Loss",      lambda v: f"{v:.6f}"),
    ("Winkler Score",  lambda v: f"{v:.6f}"),
    ("Mean VaR Level", lambda v: f"{v:.5f}"),
]

names = list(results.keys())
col_w = 16

SEP3 = "=" * (26 + col_w * len(names))
print()
print(SEP3)
print("COMPARATIVA GLOBAL 2015-2024 — 5 MODELOS")
print(SEP3)
print(f"  {'Métrica':<24}" + "".join(f"{n:>{col_w}}" for n in names))
print("  " + "-"*24 + "".join("-"*col_w for _ in names))
for metric, fmtfn in metrics_cmp:
    row = f"  {metric:<24}"
    for name in names:
        v = results[name].get(metric, np.nan)
        row += f"{fmtfn(v):>{col_w}}"
    print(row)
print(SEP3)

# Resumen PASS/FAIL
print()
print("RESUMEN DE TESTS (PASS/FAIL):")
print(f"  {'Modelo':<20} {'UC':>6}  {'CC':>6}  {'DQ':>6}  {'Válido(CC&DQ)':>14}")
print(f"  {'-'*20} {'-'*6}  {'-'*6}  {'-'*6}  {'-'*14}")
for name in names:
    r = results[name]
    valid = "YES" if r["CC"] == "PASS" and r["DQ"] == "PASS" else "NO"
    print(f"  {name:<20} {r['UC']:>6}  {r['CC']:>6}  {r['DQ']:>6}  {valid:>14}")